# Xây dựng mô hình RNN cho bài toán Part-of-Speech Tagging 

## Tiền xử lý dữ liệu 

In [32]:
from typing import List, Tuple 

def load_conllu(file_path: str) -> List[List[Tuple[str, str]]]:
    """
    Load a CoNLL-U formatted file and return a list of sentences,
    where each sentence is a list of (word, POS) tuples.
    """
    sentences = []
    with open(file_path, 'r', encoding='utf-8') as f:
        current_sentence = []
        for line in f:
            line = line.strip()
            if line == "":
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            elif line.startswith("#"):
                continue  # Skip comment lines
            else:
                parts = line.split('\t')
                if len(parts) >= 4:
                    word = parts[2]
                    pos = parts[4]
                    current_sentence.append((word, pos))
        if current_sentence:
            sentences.append(current_sentence)  # Add last sentence if file doesn't end with a newline
    return sentences

In [33]:
train_sentences = load_conllu(r"D:\NLP_DL\data\UD_English-EWT\en_ewt-ud-train.conllu")
dev_sentences = load_conllu(r"D:\NLP_DL\data\UD_English-EWT\en_ewt-ud-dev.conllu")
test_sentences = load_conllu(r"D:\NLP_DL\data\UD_English-EWT\en_ewt-ud-test.conllu")

print(f"Train size: {len(train_sentences)}")
print(f"Example first sentence: {train_sentences[0][:5]}")

Train size: 12544
Example first sentence: [('Al', 'NNP'), ('-', 'HYPH'), ('Zaman', 'NNP'), (':', ':'), ('American', 'JJ')]


In [34]:
def build_vocab(sentences: List[List[Tuple[str, str]]]) -> Tuple[dict, dict]:
  word_to_ix = {"<UNK>": 0}
  tag_to_ix = {}
  for sent in sentences:
      for word, tag in sent:
          if word not in word_to_ix:
              word_to_ix[word] = len(word_to_ix)
          if tag not in tag_to_ix:
              tag_to_ix[tag] = len(tag_to_ix)
  return word_to_ix, tag_to_ix

In [35]:
word_to_ix, tag_to_ix = build_vocab(train_sentences)

print(f" word_to_ix size: {len(word_to_ix)}")
print(f" tag_to_ix size: {len(tag_to_ix)}")

# Xem thử 10 phần tử đầu tiên
print(list(word_to_ix.items())[:10])
print(list(tag_to_ix.items()))

 word_to_ix size: 14311
 tag_to_ix size: 50
[('<UNK>', 0), ('Al', 1), ('-', 2), ('Zaman', 3), (':', 4), ('American', 5), ('force', 6), ('kill', 7), ('Shaikh', 8), ('Abdullah', 9)]
[('NNP', 0), ('HYPH', 1), (':', 2), ('JJ', 3), ('NNS', 4), ('VBD', 5), (',', 6), ('DT', 7), ('NN', 8), ('IN', 9), ('.', 10), ('-LRB-', 11), ('MD', 12), ('VB', 13), ('VBG', 14), ('PRP', 15), ('TO', 16), ('-RRB-', 17), ('VBN', 18), ('RP', 19), ('CD', 20), ('VBZ', 21), ('RB', 22), ('NNPS', 23), ('VBP', 24), ('PRP$', 25), ('CC', 26), ('_', 27), ('WP', 28), ('EX', 29), ('WDT', 30), ('RBR', 31), ('PDT', 32), ('JJR', 33), ('WRB', 34), ('JJS', 35), ('``', 36), ("''", 37), ('POS', 38), ('RBS', 39), ('WP$', 40), ('ADD', 41), ('FW', 42), ('LS', 43), ('UH', 44), ('AFX', 45), ('$', 46), ('NFP', 47), ('SYM', 48), ('GW', 49)]


## Task 2: Tạo PyTorch Dataset và DataLoader 

In [5]:
import torch

In [36]:
from torch.utils.data import Dataset, DataLoader 
from torch.nn.utils.rnn import pad_sequence

class POSDataset(Dataset):
    def __init__(self, sentences: List[List[Tuple[str, str]]], word_to_ix: dict, tag_to_ix: dict):
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        words = [word for word, tag in sentence]
        tags = [tag for word, tag in sentence]
        
        word_indices = [self.word_to_ix.get(word, self.word_to_ix["<UNK>"]) for word in words]
        tag_indices = [self.tag_to_ix[tag] for tag in tags]
        
        return torch.tensor(word_indices, dtype=torch.long), torch.tensor(tag_indices, dtype=torch.long)

In [37]:
def collate_fn(batch):
    word_seqs, tag_seqs = zip(*batch)
    word_seqs_padded = pad_sequence(word_seqs, batch_first=True, padding_value=0)
    tag_seqs_padded = pad_sequence(tag_seqs, batch_first=True, padding_value=-1)
    
    return word_seqs_padded, tag_seqs_padded

In [38]:
train_dataset = POSDataset(train_sentences, word_to_ix, tag_to_ix)
dev_dataset = POSDataset(dev_sentences, word_to_ix, tag_to_ix)
test_dataset = POSDataset(test_sentences, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

## Task 3: Xây dựng mô hình RNN 

In [39]:
import torch.nn as nn
class SimpleRNNForTokenClassification(nn.Module):
  def __init__(self, vocab_size, target_size, embedding_dim=100, hidden_dim=128):
      super(SimpleRNNForTokenClassification, self).__init__()
      
      self.embedding = nn.Embedding(vocab_size, embedding_dim)
      self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,   
        )
      self.fc = nn.Linear(hidden_dim, target_size)
  
  def forward(self, sentences):
      embed = self.embedding(sentences)  
      outputs, hidden = self.rnn(embed)
      logits = self.fc(outputs)  
      return logits

## Task 4: Huấn luyện mô hình 

### Khởi tạo mô hình, optimizer và loss function 

In [40]:
import torch.optim as optim

embedding_dim = 100
hidden_dim = 128

vocab_size = len(word_to_ix)
target_size = len(tag_to_ix)

model = SimpleRNNForTokenClassification(vocab_size, target_size, embedding_dim, hidden_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-1)

### Traning Loop 

In [ ]:
def train_model(model, train_loader, optimizer, criterion, num_epochs=10):

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_idx, (sentences, tags) in enumerate(train_loader):
            # (1) Xóa gradient cũ
            optimizer.zero_grad()
            # (2) Forward pass
            logits = model(sentences)
            # (3) Tính loss
            logits = logits.view(-1, logits.shape[-1])  
            tags = tags.view(-1)  
            loss = criterion(logits, tags)
            # (4) Backward pass và cập nhật tham số
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {avg_loss:.4f}")

In [42]:
train_model(model, train_loader, optimizer, criterion, num_epochs=5)

Epoch 1/5, Training Loss: 1.3354
Epoch 2/5, Training Loss: 0.7418
Epoch 3/5, Training Loss: 0.5783
Epoch 4/5, Training Loss: 0.4825
Epoch 5/5, Training Loss: 0.4164


## Task 5: Đánh giá mô hình 

In [43]:
def evaluate(model, data_loader):
    model.eval()
    total, correct = 0, 0

    with torch.no_grad():
        for sentences, tags in data_loader:
            logits = model(sentences)
            predictions = torch.argmax(logits, dim=-1)

            mask = tags != -1  
            total += mask.sum().item()
            correct += ((predictions == tags) & mask).sum().item()
    accuracy = correct / total if total > 0 else 0
    return accuracy

In [44]:
def train_model(model, train_loader, dev_loader, optimizer, criterion, num_epochs=5, device="cpu"):
    best_dev_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for sentences, tags in train_loader:
            sentences, tags = sentences.to(device), tags.to(device)

            optimizer.zero_grad()
            logits = model(sentences)
            loss = criterion(logits.view(-1, logits.shape[-1]), tags.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Đánh giá sau mỗi epoch
        train_acc = evaluate(model, train_loader)
        dev_acc = evaluate(model, dev_loader)
        avg_loss = total_loss / len(train_loader)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Dev Acc: {dev_acc:.4f}")

        # Lưu mô hình tốt nhất
        if dev_acc > best_dev_acc:
            best_dev_acc = dev_acc
            torch.save(model.state_dict(), "D:/NLP_DL/src/models/best_model.pt")

    print(f"\nBest Dev Accuracy: {best_dev_acc:.4f}")


In [45]:
train_model(model, train_loader, dev_loader, optimizer, criterion, num_epochs=5)

Epoch [1/5] - Loss: 0.3662 | Train Acc: 0.8910 | Dev Acc: 0.8167
Epoch [2/5] - Loss: 0.3255 | Train Acc: 0.9006 | Dev Acc: 0.8213
Epoch [3/5] - Loss: 0.2912 | Train Acc: 0.9118 | Dev Acc: 0.8217
Epoch [4/5] - Loss: 0.2630 | Train Acc: 0.9209 | Dev Acc: 0.8232
Epoch [5/5] - Loss: 0.2382 | Train Acc: 0.9291 | Dev Acc: 0.8258

Best Dev Accuracy: 0.8258


In [46]:
def predict_sentence(model, sentence, word_to_ix, ix_to_tag):
  model.eval()
  words = sentence.split()
  words = [word.lower() for word in words]
  word_indices = [word_to_ix.get(word, word_to_ix["<UNK>"]) for word in words]
  input_tensor = torch.tensor(word_indices, dtype=torch.long).unsqueeze(0)  # Thêm batch dimension

  with torch.no_grad():
      logits = model(input_tensor)
      predictions = torch.argmax(logits, dim=-1).squeeze(0)  
      predicted_tags = [ix_to_tag[ix.item()] for ix in predictions]
      return list(zip(words, predicted_tags))

In [47]:
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

model.load_state_dict(torch.load("D:/NLP_DL/src/models/best_model.pt"))

predict_sentence(model, "The cat sits on the mat", word_to_ix, ix_to_tag)

[('the', 'DT'),
 ('cat', 'NN'),
 ('sits', 'NNS'),
 ('on', 'IN'),
 ('the', 'DT'),
 ('mat', 'NN')]

In [48]:
predict_sentence(model, "The cat sit on the mat", word_to_ix, ix_to_tag)

[('the', 'DT'),
 ('cat', 'NN'),
 ('sit', 'VBD'),
 ('on', 'IN'),
 ('the', 'DT'),
 ('mat', 'NN')]